Install packages and load data

In [ ]:
!pip -q install pandas pyarrow scikit-learn gensim

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

BASE_PATH = Path("/content/drive/MyDrive/AI-Resume-Screener")

train_df = pd.read_parquet(BASE_PATH / "data/processed/baseline_train.parquet")
test_df = pd.read_parquet(BASE_PATH / "data/processed/baseline_test.parquet")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


TF-IDF baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
import joblib

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

tfidf.fit(
    pd.concat([train_df["clean_resume"], train_df["clean_jd"]])
)

resume_matrix = tfidf.transform(test_df["clean_resume"])
jd_matrix = tfidf.transform(test_df["clean_jd"])

test_df["tfidf_score"] = 1 - paired_cosine_distances(
    resume_matrix,
    jd_matrix,
)

joblib.dump(tfidf, BASE_PATH / "models/tfidf_vectorizer.joblib")

['/content/drive/MyDrive/AI-Resume-Screener/models/tfidf_vectorizer.joblib']

Train Word2Vec

In [ ]:
from gensim.models import Word2Vec

sentences = (
    train_df["clean_resume"].str.split().tolist()
    + train_df["clean_jd"].str.split().tolist()
)

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42,
)

w2v_model.save(str(BASE_PATH / "models/word2vec/word2vec.model"))

Calculate Word2Vec pair scores

In [ ]:
def document_vector(text, model):
    vectors = [
        model.wv[word]
        for word in str(text).split()
        if word in model.wv
    ]

    if not vectors:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

def cosine_score(vector_a, vector_b):
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / denominator)

test_df["word2vec_score"] = [
    cosine_score(
        document_vector(resume, w2v_model),
        document_vector(jd, w2v_model),
    )
    for resume, jd in zip(test_df["clean_resume"], test_df["clean_jd"])
]

Save baseline predictions

In [ ]:
test_df.to_parquet(
    BASE_PATH / "outputs/baseline_predictions.parquet",
    index=False,
)

display(test_df[["label", "score", "tfidf_score", "word2vec_score"]].head())

,label,score,tfidf_score,word2vec_score
0,Good Fit,1.0,0.069276,0.890795
1,No Fit,0.0,0.028743,0.648857
2,Good Fit,1.0,0.095514,0.905545
3,No Fit,0.0,0.048533,0.603353
4,No Fit,0.0,0.026576,0.642973
